# NeoOLAF DocRED — resume ONLY the failed documents + final full micro evaluation

This notebook resumes the **existing** DocRED v6.2 full-dev experiment under the exact original batch root:

`examples/RAGTreeDatasets/runs/docred_native_v5_1_dev_streaming`

It does **not** start a new experiment.

## Intended starting state

The previous full run requested **998 DocRED dev records**:

- **978 completed**
- **20 failed**

This notebook:

1. rebuilds the current aggregate **offline** from the existing saved artifacts;
2. lists the currently failed records;
3. verifies that every non-failed document is already completed/resumable;
4. reruns the v6.2 streaming scheduler with:
   - `resume_completed=True`
   - `retry_failed_documents=True`
5. therefore **completed documents are skipped** and only failed/unresolved documents are eligible for paid rerun;
6. uses the original frozen DocRED execution settings:
   - model `openai/gpt-oss-20b`
   - `DOCUMENT_WORKERS=4`
   - `LAYER_WORKERS=16`
   - frozen v5.1 scientific profile/guidance/evaluator
7. rebuilds the final aggregate from disk afterward;
8. reports the definitive DocRED relation/entity/endpoint micro metrics and runtime.

If some failed documents remain after one invocation, rerun this notebook: successfully recovered documents are then skipped and only the remaining failures are retried.


In [ ]:
from __future__ import annotations

import os
import sys
import json
import multiprocessing as mp
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents, Path(r"C:\Users\galencarmedeiro\NeoOLAF")]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the NeoOLAF repository "
        "or set the working directory to the NeoOLAF project root."
    )


def first_existing_path(label: str, candidates: list[Path]) -> Path:
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        candidate = candidate if candidate.is_absolute() else PROJECT_ROOT / candidate
        candidate = candidate.resolve()
        checked.append(candidate)
        if candidate.is_file():
            print(f"{label}={candidate}")
            return candidate
    raise FileNotFoundError(
        f"Could not find {label}. Checked:\n" + "\n".join(str(path) for path in checked)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"

for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from docred_native_batch_v6_2_dev_streaming import (
    BatchRunConfig,
    aggregate_batch_results_streaming,
    count_exact_type_records,
    read_json,
    run_batch_streaming,
)

mp.freeze_support()

print("PROJECT_ROOT =", PROJECT_ROOT)
print("TOOLS_DIR    =", TOOLS_DIR)


## Frozen DocRED paths and execution configuration

These values match the original v6.2 full-dev runner. The same batch root is critical: this is what lets the scheduler recognize and skip the already completed 978 documents.


In [ ]:
RUN_ALL_DEV_DOCUMENTS = True
EXACT_TYPE_VALUE = "dev"
START_DEV_INDEX = 0

# Only used by the helper API when run_all_documents=False.
SMOKE_DEV_DOCUMENT_LIMIT = 5

DATASET_JSONL = first_existing_path(
    "DATASET_JSONL",
    [
        NOTEBOOK_DIR / "../../../ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "RAGTree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT / "ragtree/data/preprocessed/docred_causal.jsonl",
    ],
)

# MUST stay identical to the original v6.2 full run.
BATCH_ROOT = NOTEBOOK_DIR / "runs/docred_native_v5_1_dev_streaming"

ONTOLOGY_PATH = NOTEBOOK_DIR / "ontology/docred_redocred_neoolaf_compatible.ttl"
ONTOLOGY_ORIGINAL = NOTEBOOK_DIR / "ontology/docred_redocred_original.ttl"
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/docred_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/docred_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/docred_profile_native_ablation_v5.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_docred_native_ablation_v5.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/docred_task_guidance_v5_1_frozen.json"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"

# Original v6.2 orchestration.
DOCUMENT_WORKERS = 4
LAYER_WORKERS = 16

REASONING_EFFORT = "minimal"
MAX_TOKENS = 4096
REQUEST_TIMEOUT = 120

# These are the critical resume settings.
RESUME_COMPLETED = True
RETRY_FAILED_DOCUMENTS = True

# Same retry behavior as the original runner.
DOCUMENT_ATTEMPTS = 2
RETRY_BACKOFF_SECONDS = 8.0
DOCUMENT_LAUNCH_STAGGER_SECONDS = 0.75
VERBOSE_DOCUMENTS = False
PROGRESS_EVERY = 1

# Paid execution switch.
RUN_RETRY = True

print("BATCH_ROOT =", BATCH_ROOT)
print("MODEL =", MODEL_NAME)
print("DOCUMENT_WORKERS =", DOCUMENT_WORKERS)
print("LAYER_WORKERS =", LAYER_WORKERS)
print("RESUME_COMPLETED =", RESUME_COMPLETED)
print("RETRY_FAILED_DOCUMENTS =", RETRY_FAILED_DOCUMENTS)


## Zero-cost scientific and filesystem preflight


In [ ]:
required = [
    DATASET_JSONL,
    ONTOLOGY_PATH,
    ONTOLOGY_ORIGINAL,
    RELATION_CATALOG,
    RELATION_ALIASES,
    PROFILE_PATH,
    GUIDANCE_PATH,
    TASK_GUIDANCE_PATH,
]

missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

if not BATCH_ROOT.is_dir():
    raise FileNotFoundError(
        "The original DocRED batch root does not exist. "
        "This notebook is resume-only and refuses to create a new experiment:\n"
        f"{BATCH_ROOT}"
    )

profile = read_json(PROFILE_PATH)
task_guidance = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)

corpus_scan = count_exact_type_records(
    DATASET_JSONL,
    required_type=EXACT_TYPE_VALUE,
    first_n_ids=5,
)

matching_records = int(corpus_scan["matching_records"])

print("JSONL records:", corpus_scan["total_records"])
print('Records with exact type="dev":', matching_records)
print("Ontology properties:", catalog["property_count"])
print("Allowed relation IDs:", len(task_guidance["allowed_relation_ids"]))
print("Frozen profile:", profile["profile_name"])

assert EXACT_TYPE_VALUE == "dev"
assert matching_records == 998, (
    "Expected the same 998 exact type=dev records from the original run.",
    matching_records,
)
assert catalog["property_count"] == 96
assert len(task_guidance["allowed_relation_ids"]) == 96

# Anti-cheating / gold-isolation invariants from the frozen profile.
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_docred_extraction"] is False
assert profile["anti_cheating"]["source_entity_anchoring"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False

# Freeze the execution shell.
assert DOCUMENT_WORKERS == 4
assert LAYER_WORKERS == 16
assert RESUME_COMPLETED is True
assert RETRY_FAILED_DOCUMENTS is True

print("\nPreflight: OK")
print("No API calls made.")


## Rebuild CURRENT aggregate from saved artifacts — zero API calls

This determines the exact current number of successful and failed documents before any retry.

At the original stopping point this should show **978 completed / 20 failed**.  
The notebook also remains safe if you rerun it after recovering some of those failures.


In [ ]:
before = aggregate_batch_results_streaming(
    batch_root=BATCH_ROOT,
    relation_catalog_path=RELATION_CATALOG,
)

before_summary = before["summary"]

before_requested = int(before_summary["documents_requested"])
before_completed = int(before_summary["documents_completed_or_resumed"])
before_failed = int(before_summary["documents_failed"])

print("CURRENT DOCRED STATE")
print("====================")
print("requested:", before_requested)
print("completed/resumed:", before_completed)
print("failed:", before_failed)

assert before_requested == 998, before_requested
assert before_completed + before_failed == before_requested, (
    before_completed,
    before_failed,
    before_requested,
)
assert before_completed >= 978, (
    "This resume notebook expects the previous full run or a later partial retry.",
    before_completed,
)
assert 0 <= before_failed <= 20, before_failed

failed_path = Path(before["paths"]["failed_documents_jsonl"])
print("Failures JSONL:", failed_path)

def read_jsonl(path: Path):
    rows = []
    if not path.is_file():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

failed_rows_before = read_jsonl(failed_path)

# The aggregate count is authoritative. The failure artifact should agree.
assert len(failed_rows_before) == before_failed, (
    len(failed_rows_before),
    before_failed,
)

if failed_rows_before:
    failed_df = pd.DataFrame(failed_rows_before)
    columns = [
        c for c in [
            "selection_index",
            "source_index",
            "document_id",
            "title",
            "status",
            "error_type",
            "error",
            "run_dir",
        ]
        if c in failed_df.columns
    ]
    display(failed_df[columns])
else:
    print("No failed documents remain.")

display(Markdown(
    f"**Current state:** {before_completed}/998 complete; "
    f"**{before_failed} failed documents eligible for retry**.  \n"
    "The completed documents are protected by `resume_completed=True`."
))


## Paid retry cell — ONLY failed/unresolved documents are eligible

`run_batch_streaming` still scans the dev JSONL to recover source records, but the existing batch state causes completed records to be resumed/skipped.

It does **not** pay to rerun the already-completed documents.


In [ ]:
if before_failed == 0:
    print("DocRED is already 998/998. Paid retry skipped.")
    batch = before

elif not RUN_RETRY:
    print(
        f"RUN_RETRY=False: {before_failed} failed document(s) remain. "
        "No API calls made."
    )
    batch = before

else:
    API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    config = BatchRunConfig(
        project_root=str(PROJECT_ROOT),
        ontology_path=str(ONTOLOGY_PATH),
        profile_path=str(PROFILE_PATH),
        guidance_path=str(GUIDANCE_PATH),
        relation_catalog_path=str(RELATION_CATALOG),
        relation_aliases_path=str(RELATION_ALIASES),
        model_name=MODEL_NAME,
        host=OPENROUTER_HOST,
        document_workers=DOCUMENT_WORKERS,
        layer_workers=LAYER_WORKERS,
        reasoning_effort=REASONING_EFFORT,
        max_tokens=MAX_TOKENS,
        request_timeout=REQUEST_TIMEOUT,
        resume_completed=RESUME_COMPLETED,
        retry_failed_documents=RETRY_FAILED_DOCUMENTS,
        document_attempts=DOCUMENT_ATTEMPTS,
        retry_backoff_seconds=RETRY_BACKOFF_SECONDS,
        launch_stagger_seconds=DOCUMENT_LAUNCH_STAGGER_SECONDS,
        verbose_documents=VERBOSE_DOCUMENTS,
        progress_every=PROGRESS_EVERY,
    )

    print(
        f"Retrying unresolved DocRED documents only: {before_failed} currently failed.\n"
        f"Completed documents protected: {before_completed}."
    )

    batch = run_batch_streaming(
        dataset_jsonl=DATASET_JSONL,
        task_guidance_path=TASK_GUIDANCE_PATH,
        batch_root=BATCH_ROOT,
        run_all_documents=True,
        smoke_document_limit=SMOKE_DEV_DOCUMENT_LIMIT,
        start_index=0,
        config=config,
        api_key=API_KEY,
        matching_record_count=matching_records,
        required_type=EXACT_TYPE_VALUE,
    )

    print("Retry invocation finished.")


## Rebuild final aggregate from disk — zero additional API calls

This deliberately rebuilds the aggregate again from persisted document artifacts instead of trusting only the in-memory scheduler result.


In [ ]:
final_batch = aggregate_batch_results_streaming(
    batch_root=BATCH_ROOT,
    relation_catalog_path=RELATION_CATALOG,
)

summary = final_batch["summary"]

requested = int(summary["documents_requested"])
completed = int(summary["documents_completed_or_resumed"])
failed = int(summary["documents_failed"])

print("FINAL/CURRENT DOCRED STATE")
print("==========================")
print("requested:", requested)
print("completed/resumed:", completed)
print("failed:", failed)

assert requested == 998
assert completed + failed == requested

final_metrics = {
    "documents_requested": requested,
    "documents_completed_or_resumed": completed,
    "documents_failed": failed,

    "relation_micro_precision": summary["micro_relation"]["precision"],
    "relation_micro_recall": summary["micro_relation"]["recall"],
    "relation_micro_f1": summary["micro_relation"]["f1"],

    "relation_macro_precision": summary["macro_relation"]["precision"],
    "relation_macro_recall": summary["macro_relation"]["recall"],
    "relation_macro_f1": summary["macro_relation"]["f1"],

    "entity_micro_precision": summary["micro_entity_inventory"]["precision"],
    "entity_micro_recall": summary["micro_entity_inventory"]["recall"],
    "entity_micro_f1": summary["micro_entity_inventory"]["f1"],

    "endpoint_micro_precision": summary["micro_relation_endpoint_inventory"]["precision"],
    "endpoint_micro_recall": summary["micro_relation_endpoint_inventory"]["recall"],
    "endpoint_micro_f1": summary["micro_relation_endpoint_inventory"]["f1"],

    "mean_pipeline_seconds": summary["mean_document_pipeline_seconds"],
    "median_pipeline_seconds": summary["median_document_pipeline_seconds"],
}

display(pd.DataFrame([final_metrics]))

print("\nRELATION MICRO")
print(
    f"P={summary['micro_relation']['precision']:.9f} | "
    f"R={summary['micro_relation']['recall']:.9f} | "
    f"F1={summary['micro_relation']['f1']:.9f}"
)

print("\nENTITY MICRO")
print(
    f"P={summary['micro_entity_inventory']['precision']:.9f} | "
    f"R={summary['micro_entity_inventory']['recall']:.9f} | "
    f"F1={summary['micro_entity_inventory']['f1']:.9f}"
)

print("\nRELATION ENDPOINT MICRO")
print(
    f"P={summary['micro_relation_endpoint_inventory']['precision']:.9f} | "
    f"R={summary['micro_relation_endpoint_inventory']['recall']:.9f} | "
    f"F1={summary['micro_relation_endpoint_inventory']['f1']:.9f}"
)

failed_rows_after = read_jsonl(Path(final_batch["paths"]["failed_documents_jsonl"]))

if failed_rows_after:
    print(f"\nStill failed: {len(failed_rows_after)}")
    failed_after_df = pd.DataFrame(failed_rows_after)
    columns = [
        c for c in [
            "selection_index",
            "source_index",
            "document_id",
            "title",
            "status",
            "error_type",
            "error",
            "run_dir",
        ]
        if c in failed_after_df.columns
    ]
    display(failed_after_df[columns])
    print(
        "\nRerun this notebook to retry ONLY those remaining failures. "
        "Newly recovered documents will be skipped next time."
    )
else:
    print("\nSUCCESS: DocRED is now 998/998 with zero failed documents.")


## Exact relation counts + final exports

This reads the aggregate files generated by the existing DocRED evaluator, including TP/FP/FN/predicted/gold relation totals where available.


In [ ]:
per_document_path = Path(final_batch["paths"]["per_document_csv"])
per_document = (
    pd.read_csv(per_document_path)
    if per_document_path.is_file()
    else pd.DataFrame()
)

cumulative_path = BATCH_ROOT / "aggregate_analysis/cumulative_layer_micro_evaluation.csv"
cumulative = (
    pd.read_csv(cumulative_path)
    if cumulative_path.is_file()
    else pd.DataFrame()
)

if not cumulative.empty:
    final_layer = cumulative.sort_values("layer_index").iloc[-1]
    print("FINAL LAYER RELATION COUNTS")
    for key in [
        "predicted",
        "gold",
        "true_positive",
        "false_positive",
        "false_negative",
        "precision",
        "recall",
        "f1",
    ]:
        if key in final_layer:
            print(f"{key}: {final_layer[key]}")

if not per_document.empty:
    print("\nPER-DOCUMENT RUNTIME")
    if "pipeline_seconds" in per_document.columns:
        print("documents:", len(per_document))
        print("mean:", float(per_document["pipeline_seconds"].mean()))
        print("median:", float(per_document["pipeline_seconds"].median()))
        print("min:", float(per_document["pipeline_seconds"].min()))
        print("max:", float(per_document["pipeline_seconds"].max()))

FINAL_REPORT_PATH = BATCH_ROOT / "aggregate_analysis/docred_final_resume20_report.json"
FINAL_REPORT_PATH.write_text(
    json.dumps(
        {
            "model": MODEL_NAME,
            "batch_root": str(BATCH_ROOT),
            "retry_started_from": {
                "completed": before_completed,
                "failed": before_failed,
            },
            "final": final_metrics,
            "remaining_failed_documents": failed_rows_after,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nSaved final report:", FINAL_REPORT_PATH)
print("Batch summary:", final_batch["paths"]["batch_summary"])
print("Per-document CSV:", final_batch["paths"]["per_document_csv"])
print("Per-relation CSV:", BATCH_ROOT / "aggregate_analysis/per_relation_metrics.csv")
print("Cumulative layer CSV:", cumulative_path)
print("Failures JSONL:", final_batch["paths"]["failed_documents_jsonl"])
